# KCDK season persistence and player usage
This notebook exercises package APIs against four fictional weeks and a temporary SQLite database. Business logic remains in `src/kcdk`.

In [ ]:
from pathlib import Path
import sys
import tempfile

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from kcdk.analytics import (
    consecutive_player_use, group_player_usage, member_ownership_extremes,
    member_player_usage, most_used_players_by_member, season_leaderboard,
    unanimous_weekly_selections, unique_weekly_selections,
)
from kcdk.persistence import connect_database, import_week, weekly_results

## Create a temporary database and import all mock weeks

In [ ]:
temporary_directory_context = tempfile.TemporaryDirectory(prefix='kcdk-notebook-')
temporary_directory = Path(temporary_directory_context.name)
database_path = temporary_directory / 'season.sqlite'
connection = connect_database(database_path)
mock_directory = ROOT / 'data' / 'mock'
members_path = mock_directory / 'members.csv'

summaries = [
    import_week(
        connection, mock_directory / f'season_week_{week}.csv', members_path,
        season_name='Mock 2026', season_identifier='mock-2026',
        season_year=2026, week_number=week,
        contest_name=f'Fictional Week {week}', contest_date=f'2026-09-{week:02d}',
    ).to_dict()
    for week in range(1, 5)
]
summaries

## Weekly standings and official season leaderboard

In [ ]:
weekly_results(connection, 'mock-2026')

In [ ]:
season_leaderboard(connection, 'mock-2026')

## Member and group player usage

In [ ]:
member_player_usage(connection, 'mock-2026', 'casey-north')

In [ ]:
group_player_usage(connection, 'mock-2026')

## Factual usage patterns

In [ ]:
patterns = {
    'consecutive': consecutive_player_use(connection, 'mock-2026'),
    'unanimous': unanimous_weekly_selections(connection, 'mock-2026'),
    'unique': unique_weekly_selections(connection, 'mock-2026'),
    'most_used_by_member': most_used_players_by_member(connection, 'mock-2026'),
    'ownership_extremes': member_ownership_extremes(connection, 'mock-2026'),
}
patterns

In [ ]:
connection.close()
temporary_directory_context.cleanup()
{'database_path': database_path, 'cleaned_up': not database_path.exists()}